# FLAG 2027 — NB-4: ReDimNet2 voice encoders (round 3)

One question, one variable: **does multilingual pretraining of the voice encoder help the Bangla cells?**
Two ReDimNet2-B6 checkpoints that differ ONLY in training data (MIT licence, github.com/PalabraAI/redimnet2):

| short | checkpoint | training data |
|---|---|---|
| `rdn6vox` | b6 / lm / `vox2` | VoxCeleb2 (mostly English) |
| `rdn6multi` | b6 / lm / `vb2+vox2+cnc2_v0` | VoxBlink2 + VoxCeleb2 + CN-Celeb2 (multilingual) |

Everything else is the EXP-007 recipe, and the crops reuse NB-1's `crop_plan_train.npy`, so `*_mix` runs are
directly comparable with `r2_ecapa192_mix`.

**Settings:** GPU T4, **Internet ON** (torch.hub download). **Inputs:** the raw dataset (wav), `flag2027-features`,
`flag2027-feats-v2` (crop plan). ~45-60 min total.

⚠️ ReDimNet2 has no length mask: zero padding changes its embedding (local test: cos 0.63 with 3 s of padding),
so extraction runs **one file at a time** (no batching). Slower, but exact.

Output: `feats_v3/` (new arrays + manifest), `submissions/r3_*.zip`, `*_scores.npz`, `*_matrix.npz`
(full face x voice score matrices for graph refinement), `runs.csv`, `p0_unimodal_v3.csv`.

In [ ]:
import os, sys
from pathlib import Path
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| working", WORKDIR)

In [ ]:
# Writes the helper modules next to this notebook (no .py upload needed).
import base64
open('flag_lib.py', 'wb').write(base64.b64decode(''.join('''
...
cmV0dXJuIHBhdGgK
'''.split())))
open('flag_extract.py', 'wb').write(base64.b64decode(''.join('''
...
'''.split())))
open('flag_v2.py', 'wb').write(base64.b64decode(''.join('''
...
'''.split())))
print('written:', ['flag_lib.py', 'flag_extract.py', 'flag_v2.py'])

## 1. Extract ReDimNet2-B6 (vox2) and (vb2+vox2+cnc2), one file at a time

In [ ]:
import numpy as np, pandas as pd, torch, time, json
import flag_extract as X
OUT3 = WORKDIR / "feats_v3"; OUT3.mkdir(exist_ok=True)
root = X.media_root(IN, WORKDIR / "raw")
S = X.index_splits(root)
plan = sorted(IN.rglob("crop_plan_train.npy"))
assert plan, "crop_plan_train.npy not found: attach flag2027-feats-v2"
# multilingual first: it is the question of this round, and it is the slow one (extra GroupNorm layers)
ENCS = [("b6", "vb2+vox2+cnc2_v0", "rdn6multi"), ("b6", "vox2", "rdn6vox")]
BUDGET_MIN = 0.5 if LOCAL else 150          # per encoder; the guard drops crops, then the encoder
DONE_ENC = []
for mn, ds, short in ENCS:
    t0 = time.time()
    model = X.load_redimnet2(mn, ds)
    try:
        X.extract_hub_voice(S, OUT3, short, model, crop_plan_file=plan[0], max_minutes=BUDGET_MIN,
                            probe_files=10 if LOCAL else 50)
        DONE_ENC.append(short)
    except X.EncoderTooSlow as e:
        print("SKIPPED:", e)
    del model; torch.cuda.empty_cache()
    print(f"{short}: {time.time() - t0:.0f}s")
print("extracted:", DONE_ENC)
X.write_manifest(OUT3, dict(encoders=ENCS, extracted=DONE_ENC, crop_plan=str(plan[0])))

## 2. Unimodal sanity + EXP-007 recipe on each encoder (full / Bangla-length crops)

In [ ]:
os.environ["FLAG_DATA"] = str(IN)
os.environ["FLAG_FEATS_EXTRA"] = str(OUT3)
import flag_v2 as V
V.polarity_preflight()
store = V.FeatureStore(IN)
# unimodal sanity (speaker EER on train, no training): a broken extraction shows up here immediately
rows = []
for n in ["given", "ecapa192", "rdn6vox", "rdn6multi"]:
    if store.has("voice", n):
        rows.append(dict(feat=n, **V.unimodal_eer(store.voice(n, "train"), store.spk, store.gmap)))
        C = store.voice_crops(n)
        if C is not None:
            rows.append(dict(feat=n + " (Bangla-length crop)", **V.unimodal_eer(C[0], store.spk, store.gmap)))
P0 = pd.DataFrame(rows); P0.to_csv(WORKDIR / "p0_unimodal_v3.csv", index=False); print(P0.to_string(index=False))

REC = dict(V.BASE, head="mlp", drop=0.3, emb=128, epochs=40)      # = EXP-007 recipe, same as r2_*
RUNS = {f"r3_{v}{'_mix' if m else ''}": dict(REC, voice=v, voice_view="mix" if m else "full")
        for v in ["rdn6multi", "rdn6vox"] if store.has("voice", v)
        for m in (False, True) if not m or store.voice_crops(v) is not None}
print("runs:", list(RUNS))
N_MODELS = 2 if LOCAL else 10
if LOCAL:
    RUNS = {k: dict(v, epochs=2) for k, v in RUNS.items()}
SUBS = WORKDIR / "submissions"; SUBS.mkdir(exist_ok=True)
for name, cfg in RUNS.items():
    out = SUBS / f"{name}.zip"
    if out.exists():
        print("skip (exists)", out.name); continue
    t0 = time.time()
    sc = V.dev_scores(store, cfg, n_models=N_MODELS, cca_w=0.25, matrix_path=SUBS / f"{name}_matrix.npz")
    np.savez(SUBS / f"{name}_scores.npz", **{"/".join(k): v for k, v in sc.items()})
    V.write_zip(store, out, sc); V.check_zip(store, out)
    pd.DataFrame([dict(zip=out.name, sec=round(time.time() - t0), cfg=json.dumps(cfg))]).to_csv(
        SUBS / "runs.csv", mode="a", header=not (SUBS / "runs.csv").exists(), index=False)
    print(f"{name}: {time.time() - t0:.0f}s")
print(sorted(p.name for p in SUBS.glob("*.zip")))

In [ ]:
import shutil
shutil.rmtree(WORKDIR / "raw", ignore_errors=True)      # never ship extracted media in the output
tot = sum(p.stat().st_size for p in WORKDIR.rglob("*") if p.is_file()) / 1e9
print(f"output {tot:.2f} GB. Download: submissions/ (zips + npz) and feats_v3/ (for the next rounds).")